In [2]:
import duckdb
import polars as pl
import pandas as pd
import json
import os

In [9]:
def add_hashed_payload_to_bronze(
    bronze_db_path: str,
    table_name: str,
) -> None:
    with duckdb.connect(bronze_db_path) as bronze_con:
        columns = {
            row[1]
            for row in bronze_con.execute(
                f"PRAGMA table_info('{table_name}')"
            ).fetchall()
        }

        if "hashed_payload" not in columns:
            bronze_con.execute(
                f"""
                ALTER TABLE {table_name}
                ADD COLUMN hashed_payload VARCHAR;
                """
            )

        bronze_con.execute(
            f"""
            UPDATE {table_name}
            SET hashed_payload = md5(
                CAST(payload AS VARCHAR)
            )
            WHERE hashed_payload IS NULL;
            """
        )

        bronze_con.execute(
            f"""
            ALTER TABLE {table_name}
            ALTER COLUMN hashed_payload SET NOT NULL;
            """
        )

In [12]:
def reorder_hashed_payload_column(
    bronze_db_path: str,
    table_name: str,
) -> None:
    temp_table_name = f"{table_name}_reordered"

    with duckdb.connect(bronze_db_path) as bronze_con:
        bronze_con.execute("BEGIN TRANSACTION")

        try:
            bronze_con.execute(
                f"""
                CREATE TABLE {temp_table_name} (
                    run_id VARCHAR NOT NULL,
                    project_type VARCHAR NOT NULL,
                    project_id VARCHAR NOT NULL,
                    payload JSON NOT NULL,
                    hashed_payload VARCHAR NOT NULL,
                    c_pull_timestamp_utc TIMESTAMPTZ NOT NULL,
                    PRIMARY KEY (
                        run_id,
                        project_type,
                        project_id
                    )
                );
                """
            )

            bronze_con.execute(
                f"""
                INSERT INTO {temp_table_name}
                (
                    run_id,
                    project_type,
                    project_id,
                    payload,
                    hashed_payload,
                    c_pull_timestamp_utc
                )
                SELECT
                    run_id,
                    project_type,
                    project_id,
                    payload,
                    hashed_payload,
                    c_pull_timestamp_utc
                FROM {table_name};
                """
            )

            bronze_con.execute(
                f"""
                DROP TABLE {table_name};
                """
            )

            bronze_con.execute(
                f"""
                ALTER TABLE {temp_table_name}
                RENAME TO {table_name};
                """
            )

            bronze_con.execute("COMMIT")

        except Exception:
            bronze_con.execute("ROLLBACK")
            raise

In [13]:
reorder_hashed_payload_column(bronze_db_path="/Users/admin/AroTekCodingSpace/Python-Workspace/Minecraft-Data-Platform/data/bronze/dev/https:||api.modrinth.com.duckdb",
                              table_name='modrinth_project_listings')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
add_hashed_payload_to_bronze(bronze_db_path="/Users/admin/AroTekCodingSpace/Python-Workspace/Minecraft-Data-Platform/data/bronze/dev/https:||api.modrinth.com.duckdb",
                             table_name='modrinth_project_listings')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
with duckdb.connect("/Users/admin/AroTekCodingSpace/Python-Workspace/Minecraft-Data-Platform/data/bronze/dev/https:||piston-meta.mojang.com.duckdb") as con:
    df = con.execute("SELECT * FROM ingestion_log").fetch_df()
df

,run_id,ingestion_type,api_url,project_type,status,records_processed,records_written,records_failed,records_skipped,nested_records_fetched,failed_record_ids,skipped_record_ids,error_message,start_time,end_time,duration_seconds
0,dc878dcb-6d16-4727-8c1d-19ae1418ba2a,mojang_version_manifest,https://piston-meta.mojang.com/mc/game/version...,mojang_version_manifest,failed,1,0,1,0,0,<NA>,<NA>,Empty payload returned from https://piston-met...,2026-08-25 16:57:22.363441-07:00,2026-08-25 16:57:23.024617-07:00,0.661176
1,45d92c82-1e91-48ae-acba-f1bf20da122a,mojang_version_manifest,https://piston-meta.mojang.com/mc/game/version...,mojang_version_manifest,failed,1,0,1,0,0,<NA>,<NA>,Empty payload returned from https://piston-met...,2026-08-25 17:00:17.456628-07:00,2026-08-25 17:00:17.843108-07:00,0.386480
2,15748aeb-59b4-4d81-903a-b78deffd0946,mojang_version_manifest,https://piston-meta.mojang.com/mc/game/version...,mojang_version_manifest,failed,1,0,1,0,0,<NA>,<NA>,"Binder Error: Table ""mojang_version_manifest"" ...",2026-08-25 17:00:25.845173-07:00,2026-08-25 17:00:26.076090-07:00,0.230917
3,3ccd3201-5529-40df-bef7-9e11cd4b05ac,mojang_version_manifest,https://piston-meta.mojang.com/mc/game/version...,mojang_version_manifest,success,1,1,0,0,908,<NA>,<NA>,NaN,2026-08-25 17:01:29.555595-07:00,2026-08-25 17:01:29.780129-07:00,0.224534
4,26baabc2-b283-442d-aed4-0a97adf69a94,mojang_version_manifest,https://piston-meta.mojang.com/mc/game/version...,mojang_version_manifest,success,1,1,0,0,908,<NA>,<NA>,NaN,2026-08-25 17:02:29.245835-07:00,2026-08-25 17:02:29.576753-07:00,0.330918


In [8]:
with duckdb.connect("/Users/admin/AroTekCodingSpace/Python-Workspace/Minecraft-Data-Platform/data/bronze/dev/https:||api.modrinth.com.duckdb") as con:
    df = con.execute("SELECT * FROM ingestion_log").fetch_df()
df

,run_id,ingestion_type,api_url,project_type,status,records_processed,records_written,records_failed,records_skipped,nested_records_fetched,failed_record_ids,skipped_record_ids,error_message,start_time,end_time,duration_seconds
0,21f1554d-0f8b-423c-b708-97f80453e17f,project_listings,https://api.modrinth.com/v2,mod,success,73499,72802,0,697,<NA>,<NA>,"[t3qEwmQm, t9urSl1Z, bApGjY4X, inmPbeHN, jxktW...",None,2026-08-21 19:04:29.140258-07:00,2026-08-21 19:07:50.757651-07:00,201.617393
1,21f1554d-0f8b-423c-b708-97f80453e17f,project_listings,https://api.modrinth.com/v2,modpack,success,18136,18119,0,17,<NA>,<NA>,"[rpQjGciW, jrguFbWu, 8AciRjot, OUOIQAoj, OheBI...",None,2026-08-21 19:07:50.800712-07:00,2026-08-21 19:08:12.385967-07:00,21.585255
2,21f1554d-0f8b-423c-b708-97f80453e17f,project_listings,https://api.modrinth.com/v2,resourcepack,success,33192,32972,0,220,<NA>,<NA>,"[mzDVxkyK, KXCVIm8S, JeJtzt8n, J1I1LqCu, 5X7Bq...",None,2026-08-21 19:08:12.429416-07:00,2026-08-21 19:09:03.669433-07:00,51.240017
3,21f1554d-0f8b-423c-b708-97f80453e17f,project_listings,https://api.modrinth.com/v2,shader,success,834,834,0,0,<NA>,<NA>,[],None,2026-08-21 19:09:03.712466-07:00,2026-08-21 19:09:04.597561-07:00,0.885095
4,21f1554d-0f8b-423c-b708-97f80453e17f,project_listings,https://api.modrinth.com/v2,plugin,success,17057,16849,0,208,<NA>,<NA>,"[s7cqsWbL, Prt907qL, rEq2hinY, r4AnTY0K, gnGqB...",None,2026-08-21 19:09:04.636947-07:00,2026-08-21 19:10:21.746641-07:00,77.109694
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,bc48ba6a-6ffc-49f8-92e5-6b276ff45a5e,project_versions,https://api.modrinth.com/v2,mod,success,73287,73285,2,0,960175,"[2hhxCwai, x0sIH6BC]",<NA>,None,2026-08-26 07:08:04.026587-07:00,2026-08-26 11:51:48.274291-07:00,17024.247704
66,bc48ba6a-6ffc-49f8-92e5-6b276ff45a5e,project_versions,https://api.modrinth.com/v2,modpack,success,18092,18090,2,0,167556,"[DUBosGKU, OHOB9njY]",<NA>,None,2026-08-26 11:51:48.284376-07:00,2026-08-26 13:01:05.068363-07:00,4156.783987
67,bc48ba6a-6ffc-49f8-92e5-6b276ff45a5e,project_versions,https://api.modrinth.com/v2,plugin,success,16866,16859,7,0,114565,"[GXMJQxGc, JXtiPCTt, KnjJTwnN, TCl40nBw, ltPsi...",<NA>,None,2026-08-26 13:01:05.078949-07:00,2026-08-26 14:07:16.548223-07:00,3971.469274
68,bc48ba6a-6ffc-49f8-92e5-6b276ff45a5e,project_versions,https://api.modrinth.com/v2,resourcepack,success,33634,33632,2,0,132338,"[3ZkExZDZ, bTp302p4]",<NA>,None,2026-08-26 14:07:16.559045-07:00,2026-08-26 16:19:48.401977-07:00,7951.842932


In [14]:
with duckdb.connect("/Users/admin/AroTekCodingSpace/Python-Workspace/Minecraft-Data-Platform/data/bronze/dev/https:||api.modrinth.com.duckdb") as con:
    df = con.execute("SELECT * FROM modrinth_project_listings LIMIT 10").fetch_df()
df

,run_id,project_type,project_id,payload,hashed_payload,c_pull_timestamp_utc
0,5bdf61cd-ca8a-47cc-ac3b-e00eb7fe15cc,mod,P7dR8mSH,"{""project_id"": ""P7dR8mSH"", ""project_type"": ""mo...",27b25d47cbbaa428d599eeda26fc64f4,2026-08-21 17:00:58.131666-07:00
1,5bdf61cd-ca8a-47cc-ac3b-e00eb7fe15cc,mod,AANobbMI,"{""project_id"": ""AANobbMI"", ""project_type"": ""mo...",059507daeda8cf61db99b528c50a5cb8,2026-08-21 17:00:58.131666-07:00
2,5bdf61cd-ca8a-47cc-ac3b-e00eb7fe15cc,mod,YL57xq9U,"{""project_id"": ""YL57xq9U"", ""project_type"": ""mo...",04d264d15cdfef39f4f57ecd76053f9e,2026-08-21 17:00:58.131666-07:00
3,5bdf61cd-ca8a-47cc-ac3b-e00eb7fe15cc,mod,9s6osm5g,"{""project_id"": ""9s6osm5g"", ""project_type"": ""mo...",d38fc7a2ff8761f55770c06ca24b0e1a,2026-08-21 17:00:58.131666-07:00
4,5bdf61cd-ca8a-47cc-ac3b-e00eb7fe15cc,mod,NNAgCjsB,"{""project_id"": ""NNAgCjsB"", ""project_type"": ""mo...",ed6aeca0c6830bb47ff78477cc6dc7f1,2026-08-21 17:00:58.131666-07:00
5,5bdf61cd-ca8a-47cc-ac3b-e00eb7fe15cc,mod,uXXizFIs,"{""project_id"": ""uXXizFIs"", ""project_type"": ""mo...",cb9292fb9c78d600b3bf9c01c3bbea3d,2026-08-21 17:00:58.131666-07:00
6,5bdf61cd-ca8a-47cc-ac3b-e00eb7fe15cc,mod,mOgUt4GM,"{""project_id"": ""mOgUt4GM"", ""project_type"": ""mo...",435839f4e431acf9f964df975a99e803,2026-08-21 17:00:58.131666-07:00
7,5bdf61cd-ca8a-47cc-ac3b-e00eb7fe15cc,mod,gvQqBUqZ,"{""project_id"": ""gvQqBUqZ"", ""project_type"": ""mo...",61db5d62b474c825e993bf3599627fed,2026-08-21 17:00:58.131666-07:00
8,5bdf61cd-ca8a-47cc-ac3b-e00eb7fe15cc,mod,5ZwdcRci,"{""project_id"": ""5ZwdcRci"", ""project_type"": ""mo...",d070c81000daf51eadd6b287389e1c71,2026-08-21 17:00:58.131666-07:00
9,5bdf61cd-ca8a-47cc-ac3b-e00eb7fe15cc,mod,1eAoo2KR,"{""project_id"": ""1eAoo2KR"", ""project_type"": ""mo...",703ad3592c9b1a6d20867cc1235733f7,2026-08-21 17:00:58.131666-07:00


In [15]:
payload = json.loads(df["payload"].iloc[0])

print(
    json.dumps(
        payload,
        indent=4,
    )
)

{
    "project_id": "P7dR8mSH",
    "project_type": "mod",
    "all_project_types": [
        "mod"
    ],
    "slug": "fabric-api",
    "author": "modmuss50",
    "author_id": "JZA4dW8o",
    "organization": null,
    "organization_id": null,
    "title": "Fabric API",
    "description": "Lightweight and modular API providing common hooks and intercompatibility measures utilized by mods using the Fabric toolchain.",
    "categories": [
        "fabric",
        "library"
    ],
    "display_categories": [
        "fabric",
        "library"
    ],
    "versions": [
        "18w49a",
        "18w50a",
        "19w02a",
        "19w03a",
        "19w03c",
        "19w04a",
        "19w04b",
        "19w05a",
        "19w06a",
        "19w07a",
        "19w08a",
        "19w08b",
        "19w11a",
        "19w11b",
        "19w12a",
        "19w12b",
        "19w13a",
        "19w13b",
        "19w14a",
        "19w14b",
        "1.14-pre1",
        "1.14-pre3",
        "1.14",
        "